In [ ]:
# Setup: no new library

In [1]:
import os
import sys
 
# run_query.py lives in week03/session_3_1/, next to this folder
sys.path.insert(0, os.path.abspath(os.path.join("..", "session_3_1")))
 
import duckdb
from run_query import connect_to_repo, find_repo_root, run_sql
 
con = connect_to_repo()
REPO_ROOT = find_repo_root()
# COPY creates files, not folders: make the output folders once
os.makedirs(os.path.join(REPO_ROOT, "data", "processed", "library"),
            exist_ok=True)
print("DuckDB", duckdb.__version__)

DuckDB 2.0.0


In [ ]:
# 2. Query a File Without Importing It

In [2]:
TOY = os.path.join(REPO_ROOT, "week03", "session_3_3", "toy")
os.makedirs(TOY, exist_ok=True)
with open(os.path.join(TOY, "canteen_2026_01.csv"), "w") as f:
    f.write("order_id,item,qty,price\n"
            "A01,Masala dosa,2,60.00\n"
            "A02,Filter coffee,3,20.00\n"
            "A03,Masala dosa,1,60.00\n")
with open(os.path.join(TOY, "canteen_2026_02.csv"), "w") as f:
    f.write("order_id,item,qty,price\n"
            "B01,Filter coffee,4,20.00\n"
            "B02,Idli,2,30.00\n")
print(sorted(os.listdir(TOY)))

['canteen_2026_01.csv', 'canteen_2026_02.csv']


In [3]:
run_sql("""
SELECT * FROM 'week03/session_3_3/toy/canteen_2026_01.csv';
""", con)

┌──────────┬───────────────┬───────┬────────┐
│ order_id │     item      │  qty  │ price  │
│ varchar  │    varchar    │ int64 │ double │
├──────────┼───────────────┼───────┼────────┤
│ A01      │ Masala dosa   │     2 │   60.0 │
│ A02      │ Filter coffee │     3 │   20.0 │
│ A03      │ Masala dosa   │     1 │   60.0 │
└──────────┴───────────────┴───────┴────────┘



In [4]:
run_sql("""
SELECT parse_filename(filename) AS file,
       COUNT(*) AS orders,
       SUM(qty * price) AS revenue
FROM read_csv('week03/session_3_3/toy/canteen_*.csv', filename = true)
GROUP BY file
ORDER BY file;
""", con)

┌─────────────────────┬────────┬─────────┐
│        file         │ orders │ revenue │
│       varchar       │ int64  │ double  │
├─────────────────────┼────────┼─────────┤
│ canteen_2026_01.csv │      3 │   240.0 │
│ canteen_2026_02.csv │      2 │   140.0 │
└─────────────────────┴────────┴─────────┘



In [5]:
run_sql("""
SELECT COUNT(*) AS reports FROM 'data/raw/adverse_events_10k.csv';
""", con)

┌─────────┐
│ reports │
│  int64  │
├─────────┤
│    6218 │
└─────────┘



In [ ]:
# 3. DuckDB Guesses the Types: Check Them

In [6]:
run_sql("""
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM 'data/raw/adverse_events_10k.csv');
""", con)

┌──────────────────┬─────────────┐
│   column_name    │ column_type │
│     varchar      │   varchar   │
├──────────────────┼─────────────┤
│ safety_report_id │ BIGINT      │
│ drug_name        │ VARCHAR     │
│ reaction         │ VARCHAR     │
│ serious          │ BIGINT      │
│ patient_age      │ BIGINT      │
│ patient_age_unit │ BIGINT      │
│ country          │ VARCHAR     │
│ event_date       │ BIGINT      │
└──────────────────┴─────────────┘



In [7]:
import textwrap
 
try:
    run_sql("""
    SELECT strptime(event_date, '%Y%m%d') AS report_date
    FROM 'data/raw/adverse_events_10k.csv'
    LIMIT 1;
    """, con)
except duckdb.Error as error:   # show the message, wrapped to 80
    message = type(error).__name__ + ": " + str(error).splitlines()[0]
    print(textwrap.fill(message, 80))

BinderException: Binder Error: No function matches the given name and argument
types 'strptime(BIGINT, STRING_LITERAL)'. You might need to add explicit type
casts.


In [8]:
run_sql("""
CREATE OR REPLACE VIEW events AS
SELECT *,
       CAST(strptime(event_date, '%Y%m%d') AS DATE) AS report_date
FROM read_csv('data/raw/adverse_events_10k.csv',
              types = {'safety_report_id': 'VARCHAR',
                       'serious': 'VARCHAR',
                       'patient_age_unit': 'VARCHAR',
                       'event_date': 'VARCHAR'});
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM events)
WHERE column_name IN ('safety_report_id', 'serious', 'event_date',
                      'report_date');
""", con)

┌──────────────────┬─────────────┐
│   column_name    │ column_type │
│     varchar      │   varchar   │
├──────────────────┼─────────────┤
│ safety_report_id │ VARCHAR     │
│ serious          │ VARCHAR     │
│ event_date       │ VARCHAR     │
│ report_date      │ DATE        │
└──────────────────┴─────────────┘



In [ ]:
# 4. Reading JSON

In [9]:
import json
 
with open(os.path.join(REPO_ROOT, "data", "raw",
                       "adverse_events_1000.json")) as f:
    records = json.load(f)
print(len(records), "records")
print(json.dumps(records[0], indent=2))
print(sum("patient_age" in r for r in records), "records have patient_age")

1000 records
{
  "safety_report_id": "10003302",
  "drug_name": "LYRICA",
  "reaction": "Drug ineffective",
  "serious": "2",
  "country": "US",
  "event_date": "20140312"
}
606 records have patient_age


In [10]:
run_sql("""
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM 'data/raw/adverse_events_1000.json');
SELECT COUNT(*) AS reports, COUNT(patient_age) AS with_age
FROM 'data/raw/adverse_events_1000.json';
""", con)

-- Result 1
┌──────────────────┬─────────────┐
│   column_name    │ column_type │
│     varchar      │   varchar   │
├──────────────────┼─────────────┤
│ safety_report_id │ VARCHAR     │
│ drug_name        │ VARCHAR     │
│ reaction         │ VARCHAR     │
│ serious          │ VARCHAR     │
│ country          │ VARCHAR     │
│ event_date       │ VARCHAR     │
│ patient_age      │ VARCHAR     │
└──────────────────┴─────────────┘

-- Result 2
┌─────────┬──────────┐
│ reports │ with_age │
│  int64  │  int64   │
├─────────┼──────────┤
│    1000 │      606 │
└─────────┴──────────┘



In [ ]:
# 5. Tables, Views and CTAS

In [11]:
run_sql("""
CREATE OR REPLACE VIEW orders_live AS
SELECT * FROM 'week03/session_3_3/toy/canteen_2026_01.csv';
CREATE OR REPLACE TABLE orders_snapshot AS
SELECT * FROM 'week03/session_3_3/toy/canteen_2026_01.csv';
SELECT (SELECT COUNT(*) FROM orders_live)     AS view_rows,
       (SELECT COUNT(*) FROM orders_snapshot) AS table_rows;
""", con)

┌───────────┬────────────┐
│ view_rows │ table_rows │
│   int64   │   int64    │
├───────────┼────────────┤
│         3 │          3 │
└───────────┴────────────┘



In [12]:
with open(os.path.join(TOY, "canteen_2026_01.csv"), "a") as f:
    f.write("A04,Idli,3,30.00\n")   # "a" adds a line at the end
 
run_sql("""
SELECT (SELECT COUNT(*) FROM orders_live)     AS view_rows,
       (SELECT COUNT(*) FROM orders_snapshot) AS table_rows;
""", con)

┌───────────┬────────────┐
│ view_rows │ table_rows │
│   int64   │   int64    │
├───────────┼────────────┤
│         4 │          3 │
└───────────┴────────────┘



In [13]:
db_path = os.path.join(REPO_ROOT, "week03", "session_3_3", "toy",
                       "canteen.duckdb")
if os.path.exists(db_path):
    os.remove(db_path)          # start fresh every time you run this
 
disk = duckdb.connect(db_path)  # a database stored in one file
disk.execute("CREATE TABLE menu AS SELECT 'Idli' AS item, 30 AS price")
disk.close()
 
again = duckdb.connect(db_path)  # a new connection to the same file
print(again.sql("SELECT * FROM menu").fetchall())
again.close()

[('Idli', 30)]


In [ ]:
# 6. Writing Files with COPY

In [14]:
try:
    run_sql("""
    COPY orders_snapshot TO 'data/processed/orders.csv';
    """, con)
except duckdb.Error as error:   # show the message, wrapped to 80
    message = type(error).__name__ + ": " + str(error).splitlines()[0]
    print(textwrap.fill(message, 80))
print("This notebook runs in:", os.path.relpath(os.getcwd(), REPO_ROOT))

IOException: IO Error: Cannot open file "data/processed/orders.csv": No such
file or directory
This notebook runs in: week03/session_3_3


In [15]:
run_sql("""
CREATE OR REPLACE MACRO repo_file(p) AS
    current_setting('file_search_path') || '/' || p;
COPY orders_snapshot TO (repo_file('data/processed/orders.csv'));
COPY orders_snapshot TO (repo_file('data/processed/orders.parquet'))
    (FORMAT parquet);
SELECT * FROM 'data/processed/orders.parquet';
""", con)
print(sorted(f for f in os.listdir(os.path.join(REPO_ROOT, "data",
                                                "processed"))
             if f.startswith("orders")))

┌──────────┬───────────────┬───────┬────────┐
│ order_id │     item      │  qty  │ price  │
│ varchar  │    varchar    │ int64 │ double │
├──────────┼───────────────┼───────┼────────┤
│ A01      │ Masala dosa   │     2 │   60.0 │
│ A02      │ Filter coffee │     3 │   20.0 │
│ A03      │ Masala dosa   │     1 │   60.0 │
└──────────┴───────────────┴───────┴────────┘

['orders.csv', 'orders.parquet']


In [ ]:
# 7. Parquet: Columnar Storage You Can See

In [16]:
run_sql("""
CREATE OR REPLACE TABLE events_clean AS
SELECT safety_report_id, drug_name, reaction, serious,
       CAST(patient_age AS INTEGER) AS patient_age,
       patient_age_unit, country, report_date
FROM events
ORDER BY safety_report_id;
COPY events_clean TO (repo_file('data/processed/events_clean.parquet'))
    (FORMAT parquet);
SELECT parse_filename(filename) AS file, size AS bytes
FROM read_blob(['data/raw/adverse_events_10k.csv',
                'data/processed/events_clean.parquet'])
ORDER BY bytes DESC;
""", con)

┌────────────────────────┬────────┐
│          file          │ bytes  │
│        varchar         │ int64  │
├────────────────────────┼────────┤
│ adverse_events_10k.csv │ 353198 │
│ events_clean.parquet   │ 110873 │
└────────────────────────┴────────┘



In [17]:
run_sql("""
SELECT path_in_schema AS column_name,
       total_compressed_size AS bytes_on_disk
FROM parquet_metadata('data/processed/events_clean.parquet')
ORDER BY bytes_on_disk DESC, column_name;
""", con)

┌──────────────────┬───────────────┐
│   column_name    │ bytes_on_disk │
│     varchar      │     int64     │
├──────────────────┼───────────────┤
│ reaction         │         49401 │
│ safety_report_id │         24500 │
│ drug_name        │         21586 │
│ patient_age      │          4237 │
│ country          │          4021 │
│ report_date      │          1287 │
│ patient_age_unit │          1214 │
│ serious          │           942 │
└──────────────────┴───────────────┘



In [18]:
run_sql("""
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM 'data/processed/events_clean.parquet')
WHERE column_name IN ('safety_report_id', 'serious', 'patient_age',
                      'report_date');
""", con)

┌──────────────────┬─────────────┐
│   column_name    │ column_type │
│     varchar      │   varchar   │
├──────────────────┼─────────────┤
│ safety_report_id │ VARCHAR     │
│ serious          │ VARCHAR     │
│ patient_age      │ INTEGER     │
│ report_date      │ DATE        │
└──────────────────┴─────────────┘



In [ ]:
# 8. DuckDB and pandas Together

In [19]:
top = con.sql("""
SELECT drug_name, COUNT(*) AS reports
FROM 'data/processed/events_clean.parquet'
GROUP BY drug_name
ORDER BY reports DESC, drug_name
LIMIT 3
""").df()
print(type(top).__name__)
print(top.to_string(index=False))
print(con.sql("SELECT COUNT(*) FROM events_clean").fetchall())

DataFrame
drug_name  reports
 DUPIXENT      621
CABOMETYX      170
  SKYRIZI      139
[(6218,)]


In [20]:
con.sql("SELECT SUM(reports) AS top3_reports FROM top").show()

┌──────────────┐
│ top3_reports │
│    int128    │
├──────────────┤
│          930 │
└──────────────┘



In [21]:
try:
    con.sql("SELECT 42").fetchone()
except AttributeError as error:
    print(textwrap.fill("AttributeError: " + str(error), 80))

AttributeError: Relations no longer have fetchone(): a relation keeps no result
between calls, so every call runs it again. Use rel.limit(1).fetchall() instead.


In [ ]:
# 9. Putting It Together: The Lab

In [23]:

 
CREATE OR REPLACE c AS
    current_setting('file_search_path') || '/' || p;
 
CREATE OR REPLACE VIEW events AS ...   -- the view from Section 3
 
CREATE OR REPLACE TABLE answer AS
SELECT COUNT(*) AS reports,
       MIN(report_date) AS first_day,
       MAX(report_date) AS last_day,
       COUNT(DISTINCT report_date) AS sampled_days,
       COUNT(DISTINCT drug_name) AS drugs,
       COUNT(DISTINCT country) AS countries,
       ROUND(100.0 * COUNT(*) FILTER (WHERE serious = '1')
             / COUNT(serious), 1) AS pct_serious
FROM events;
 
COPY answer
TO (repo_file('data/processed/library/01_report_profile.parquet'))
    (FORMAT parquet);
 
SELECT * FROM answer;

SyntaxError: invalid syntax (924721500.py, line 1)